# 실습 1. 피싱 웹사이트 분류 → 임베딩 RAG → 빠른 Qwen 보고서

## 사용 데이터
`Exercise1. phishing_websites_sample.csv`

## 빠른 실행 버전

```text
CSV 업로드
→ Random Forest 분류
→ 피싱 확률 상위 5건 저장
→ SentenceTransformer 임베딩 검색
→ Qwen2.5-0.5B로 간결한 분석 보고서 생성
→ Python으로 HTML 대시보드 즉시 생성
```


In [1]:
# ============================================================
# 1. 라이브러리 설치
# ============================================================

!pip -q install sentence-transformers transformers accelerate

In [2]:
# ============================================================
# 2. 라이브러리 불러오기
# ============================================================

import re
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from google.colab import files
from IPython.display import display, HTML

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)



라이브러리 로드 완료
GPU 사용 가능 여부: False


In [3]:
# ============================================================
# 3. Exercise1 CSV 업로드
# ============================================================

print("Exercise1. phishing_websites_sample.csv 파일을 업로드하세요.")

uploaded = files.upload()

csv_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".csv")
]

if not csv_files:
    raise ValueError("CSV 파일이 업로드되지 않았습니다.")

file_name = csv_files[0]
df = pd.read_csv(file_name)

print("업로드 파일:", file_name)
print("데이터 크기:", df.shape)
display(df.head())

Exercise1. phishing_websites_sample(4).csv 파일을 업로드하세요.


Saving Exercise1. phishing_websites_sample.csv to Exercise1. phishing_websites_sample.csv
업로드 파일: Exercise1. phishing_websites_sample.csv
데이터 크기: (1000, 31)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,port,HTTPS_token,Request_URL,URL_of_Anchor,Links_in_tags,SFH,Submitting_to_email,Abnormal_URL,Redirect,on_mouseover,RightClick,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,label
0,-1,-1,-1,1,-1,-1,-1,-1,1,1,1,1,-1,0,-1,-1,1,1,0,1,-1,1,1,-1,1,-1,1,1,0,-1,phishing
1,1,-1,1,1,1,-1,1,-1,1,1,1,1,-1,-1,0,-1,1,1,0,1,1,1,1,-1,-1,-1,-1,1,1,-1,phishing
2,1,-1,1,1,1,-1,0,0,1,1,1,1,-1,0,-1,-1,1,1,0,1,1,1,1,1,-1,-1,-1,1,1,1,phishing
3,1,-1,1,1,1,-1,-1,1,-1,1,1,1,1,0,0,-1,1,1,0,1,1,1,1,1,-1,0,-1,-1,1,1,phishing
4,1,-1,1,1,1,-1,1,1,-1,1,1,1,1,0,0,-1,1,1,0,1,1,1,1,-1,1,1,-1,1,0,1,normal


In [4]:
# ============================================================
# 4. 데이터 구조 검증
# ============================================================
# 잘못된 CSV를 올렸을 때 뒤에서 이해하기 어려운 오류가 나지 않도록
# Exercise1 데이터에 필요한 핵심 컬럼이 존재하는지 먼저 검사합니다.
# ============================================================

required_columns = [
    "having_IP_Address",
    "URL_Length",
    "SSLfinal_State",
    "Prefix_Suffix",
    "Abnormal_URL",
    "label"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Exercise1 피싱 웹사이트 CSV가 아닙니다.\n"
        f"누락된 핵심 컬럼: {missing_columns}\n"
        f"현재 컬럼: {df.columns.tolist()}"
    )

print("Exercise1 CSV 구조 확인 완료")
print("전체 컬럼 수:", len(df.columns))
display(df["label"].value_counts())

Exercise1 CSV 구조 확인 완료
전체 컬럼 수: 31


,count
label,
phishing,500
normal,500


In [5]:
# ============================================================
# 5. Feature와 Label 분리
# ============================================================

X = df.drop(columns=["label"]).copy()
raw_label = df["label"].copy()

# 입력값을 숫자로 변환하고 결측값을 중앙값으로 처리
X = X.apply(pd.to_numeric, errors="coerce")

for column in X.columns:
    median_value = X[column].median()

    if pd.isna(median_value):
        median_value = 0

    X[column] = X[column].fillna(median_value)


def normalize_label(value):
    text = str(value).strip().lower()

    # 업로드된 샘플은 문자열 phishing / legitimate 형식
    if text in {"phishing", "phish", "malicious", "피싱", "악성"}:
        return 1

    if text in {"legitimate", "normal", "benign", "정상"}:
        return 0

    # 숫자 라벨 데이터도 처리
    # 일반적인 UCI 피싱 웹사이트 데이터에서는 -1=phishing, 1=legitimate
    try:
        number = float(text)

        if number == -1:
            return 1

        if number == 1:
            return 0

        if number == 0:
            return 0

    except ValueError:
        pass

    raise ValueError(f"해석할 수 없는 label 값: {value}")


y = raw_label.apply(normalize_label)

print("Feature 크기:", X.shape)
print("변환된 라벨 분포:")
display(y.value_counts().rename(index={0: "정상", 1: "피싱"}))

Feature 크기: (1000, 30)
변환된 라벨 분포:


,count
label,
피싱,500
정상,500


In [6]:
# ============================================================
# 6. 학습/평가 데이터 분리
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("학습 데이터:", X_train.shape)
print("평가 데이터:", X_test.shape)

학습 데이터: (700, 30)
평가 데이터: (300, 30)


In [7]:
# ============================================================
# 7. Random Forest 모델 학습
# ============================================================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("모델 학습 완료")

모델 학습 완료


In [8]:
# ============================================================
# 8. 모델 평가
# ============================================================

y_pred = model.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred, zero_division=0):.4f}")

print("\n[혼동행렬]")
print(confusion_matrix(y_test, y_pred))

print("\n[분류 보고서]")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["정상 웹사이트", "피싱 웹사이트"],
        zero_division=0
    )
)

Accuracy : 0.9367
Precision: 0.9226
Recall   : 0.9533
F1-score : 0.9377

[혼동행렬]
[[138  12]
 [  7 143]]

[분류 보고서]
              precision    recall  f1-score   support

     정상 웹사이트       0.95      0.92      0.94       150
     피싱 웹사이트       0.92      0.95      0.94       150

    accuracy                           0.94       300
   macro avg       0.94      0.94      0.94       300
weighted avg       0.94      0.94      0.94       300



In [9]:
# ============================================================
# 9. 피싱 확률 상위 5건 추출 및 CSV 저장
# ============================================================

all_prediction = model.predict(X)
all_probability = model.predict_proba(X)[:, 1]

result_df = df.copy()
result_df["predicted_label"] = all_prediction
result_df["prediction_name"] = np.where(
    all_prediction == 1,
    "피싱 웹사이트",
    "정상 웹사이트"
)
result_df["phishing_probability"] = all_probability

top5_df = (
    result_df
    .sort_values("phishing_probability", ascending=False)
    .head(5)
    .copy()
    .reset_index(drop=True)
)

top5_df.insert(
    0,
    "event_id",
    [f"WEB-PHISH-{i:03d}" for i in range(1, 6)]
)

top5_df["detected_at"] = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

top5_df.to_csv(
    "exercise1_phishing_top5.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    top5_df[
        [
            "event_id",
            "prediction_name",
            "phishing_probability",
            "having_IP_Address",
            "URL_Length",
            "SSLfinal_State",
            "Prefix_Suffix",
            "Abnormal_URL"
        ]
    ]
)

,event_id,prediction_name,phishing_probability,having_IP_Address,URL_Length,SSLfinal_State,Prefix_Suffix,Abnormal_URL
0,WEB-PHISH-001,피싱 웹사이트,1.0,1,-1,-1,-1,1
1,WEB-PHISH-002,피싱 웹사이트,1.0,1,-1,0,-1,1
2,WEB-PHISH-003,피싱 웹사이트,1.0,-1,-1,0,-1,1
3,WEB-PHISH-004,피싱 웹사이트,1.0,-1,-1,-1,-1,-1
4,WEB-PHISH-005,피싱 웹사이트,1.0,-1,-1,0,-1,1


## 센터형 RAG 지식베이스

기존처럼 짧은 대응문서 몇 개만 사용하는 대신, 센터에서 지속적으로 축적·관리할 수 있는 지식항목으로 구성합니다.

각 지식문서는 다음 필드를 포함합니다.

- `doc_type`: 침해사고 사례, 대응절차, 분석 체크리스트, 보고기준
- `attack_category`: 공격 유형
- `summary`: 사고 또는 절차 개요
- `detection_points`: 탐지·판단 근거
- `response_steps`: 초동조치 및 대응절차
- `evidence_to_collect`: 확보해야 할 증적
- `escalation_criteria`: 보고 및 상급부대 격상 기준
- `lessons_learned`: 유사 사고 교훈
- `owner`: 담당 기능
- `updated_at`: 최종 갱신일

노트북 실행 시 기본 지식베이스가 CSV로 저장되므로, 이후 센터 내부 승인자료로 내용을 추가·수정하여 재사용할 수 있습니다.

In [10]:
# ============================================================
# 10. 센터형 피싱 웹사이트 RAG 지식베이스
# ============================================================
# 실제 운용 시에는 아래 일반화 예시를
# 센터에서 승인한 침해사고 보고서, 대응 SOP, 분석 체크리스트,
# IOC 분석자료 등으로 교체하거나 계속 추가할 수 있습니다.
#
# 보안 주의:
# - 실제 작전망 IP, 계정, 장비명, 취약점 세부정보 등은
#   승인된 내부 환경에서만 관리해야 합니다.
# - 본 실습 데이터는 교육용 일반화 예시입니다.
# ============================================================

knowledge_documents = [
    {
        "doc_id": "WEB-IR-001",
        "doc_type": "침해사고 사례",
        "title": "공식 로그인 페이지 위장 계정탈취 사고",
        "attack_category": "Credential Phishing",
        "summary": (
            "정상 기관의 로그인 화면을 모방한 피싱 페이지가 배포되어 "
            "사용자의 계정과 비밀번호 입력을 유도한 사례이다."
        ),
        "detection_points": (
            "공식 도메인과 철자가 유사한 신규 도메인, 비정상 하이픈 사용, "
            "로그인 폼 존재, 짧은 도메인 등록기간, 인증서 도메인 불일치를 확인한다."
        ),
        "response_steps": (
            "접속 차단, URL 및 도메인 차단목록 등록, 피싱 페이지 캡처, "
            "노출 계정 비밀번호 초기화, 로그인 세션 종료, MFA 재점검을 수행한다."
        ),
        "evidence_to_collect": (
            "최종 URL, 리다이렉션 체인, DNS 조회 결과, WHOIS, 인증서 정보, "
            "페이지 소스, 접속 로그, 계정 로그인 이력을 수집한다."
        ),
        "escalation_criteria": (
            "군 관련 계정 입력 정황, 다수 사용자 접속, 실제 계정 사용 성공, "
            "외부 유출 또는 추가 침투 정황이 있으면 즉시 상급 보고한다."
        ),
        "lessons_learned": (
            "도메인 문자열만으로 판단하지 말고 인증서, 등록정보, 페이지 행위와 "
            "계정 로그인 이력을 함께 분석해야 한다."
        ),
        "owner": "보안관제/침해대응",
        "updated_at": "2026-08-01",
        "keywords": "로그인 계정탈취 유사도메인 인증서 비밀번호"
    },
    {
        "doc_id": "WEB-IR-002",
        "doc_type": "침해사고 사례",
        "title": "IP 주소 직접 접속형 피싱 페이지",
        "attack_category": "IP-based Phishing",
        "summary": (
            "도메인 대신 공인 IP 주소를 직접 사용하여 가짜 인증 페이지를 제공한 사례이다."
        ),
        "detection_points": (
            "URL 호스트가 IP 주소인지, 비표준 포트가 사용되는지, "
            "인증서가 없거나 IP와 일치하지 않는지 확인한다."
        ),
        "response_steps": (
            "IP 차단, 호스팅 사업자 및 평판 확인, 동일 IP의 연관 도메인 조사, "
            "접속 단말의 브라우저·프록시 로그를 분석한다."
        ),
        "evidence_to_collect": (
            "IP 주소, 포트, HTTP 응답 헤더, 페이지 캡처, TLS 인증서, "
            "DNS Passive 자료, 접속 사용자 및 시간정보를 수집한다."
        ),
        "escalation_criteria": (
            "다수 내부 단말 접속, 자격증명 입력, 악성파일 다운로드 또는 "
            "지휘통제 관련 시스템 사칭 시 즉시 격상한다."
        ),
        "lessons_learned": (
            "IP 직접 접속은 정상 관리페이지에도 존재할 수 있으므로 "
            "페이지 목적, 인증 요구, 호스팅 평판을 함께 확인한다."
        ),
        "owner": "보안관제",
        "updated_at": "2026-08-01",
        "keywords": "IP주소 URL 비표준포트 인증페이지"
    },
    {
        "doc_id": "WEB-IR-003",
        "doc_type": "침해사고 사례",
        "title": "단축 URL 및 다단계 리다이렉션 피싱",
        "attack_category": "Redirect Phishing",
        "summary": (
            "단축 URL과 여러 중간 페이지를 이용해 최종 피싱 주소를 숨긴 사례이다."
        ),
        "detection_points": (
            "단축 URL 서비스 사용, 3회 이상 리다이렉션, 국가·도메인 급변, "
            "최종 페이지에서 계정 또는 개인정보 입력 요구 여부를 확인한다."
        ),
        "response_steps": (
            "격리 분석환경에서 리다이렉션 전체 경로 확인, 각 중간 URL 차단, "
            "최종 목적지의 페이지 및 네트워크 행위를 분석한다."
        ),
        "evidence_to_collect": (
            "HTTP 3xx 응답, Location 헤더, 전체 URL 체인, 각 도메인의 DNS·WHOIS, "
            "최종 페이지 소스와 다운로드 파일을 확보한다."
        ),
        "escalation_criteria": (
            "군 관련 문구 또는 기관 사칭, 대규모 배포, 계정입력 성공 정황이 있으면 격상한다."
        ),
        "lessons_learned": (
            "최초 URL만 차단하면 중간·최종 주소가 남을 수 있으므로 "
            "리다이렉션 체인 전체를 IOC로 관리해야 한다."
        ),
        "owner": "침해대응",
        "updated_at": "2026-08-01",
        "keywords": "단축URL 리다이렉션 Location 최종URL"
    },
    {
        "doc_id": "WEB-IR-004",
        "doc_type": "침해사고 사례",
        "title": "HTTPS 인증서를 사용한 정상사이트 위장",
        "attack_category": "HTTPS Phishing",
        "summary": (
            "공격자가 무료 인증서를 발급받아 HTTPS 자물쇠 표시를 악용한 피싱 사례이다."
        ),
        "detection_points": (
            "HTTPS 사용 여부가 아니라 인증서의 발급대상, 도메인 일치, 발급시점, "
            "유효기간과 사이트 내용의 관계를 확인한다."
        ),
        "response_steps": (
            "인증서 체인 및 발급기관 확인, 인증서 투명성 로그 검색, "
            "연관 도메인 조사와 URL 차단을 수행한다."
        ),
        "evidence_to_collect": (
            "인증서 원문, Subject·SAN, 발급일, 만료일, 지문, "
            "TLS 핸드셰이크 정보와 페이지 화면을 수집한다."
        ),
        "escalation_criteria": (
            "기관명 포함 인증서, 군 관련 시스템 사칭, 다수 접속 발생 시 보고를 격상한다."
        ),
        "lessons_learned": (
            "HTTPS는 암호화 여부를 의미할 뿐 사이트 신뢰성을 보장하지 않는다."
        ),
        "owner": "보안관제",
        "updated_at": "2026-08-01",
        "keywords": "HTTPS SSL TLS 인증서 SAN 피싱"
    },
    {
        "doc_id": "WEB-IR-005",
        "doc_type": "침해사고 사례",
        "title": "브랜드 철자 변조 및 하위도메인 위장",
        "attack_category": "Typosquatting",
        "summary": (
            "정상 기관명과 유사한 철자 또는 긴 하위도메인을 사용해 "
            "사용자가 공식 주소로 오인하도록 한 사례이다."
        ),
        "detection_points": (
            "문자 치환, 하이픈 삽입, 숫자 혼용, 긴 하위도메인, "
            "공식 도메인 문자열이 경로 또는 서브도메인에만 존재하는지 확인한다."
        ),
        "response_steps": (
            "공식 도메인과 문자열 비교, 등록일·등록기관 조사, "
            "유사 도메인 추가 탐색 및 차단을 수행한다."
        ),
        "evidence_to_collect": (
            "도메인 문자열, WHOIS, DNS 레코드, 인증서 투명성 로그, "
            "검색엔진 노출 여부와 페이지 캡처를 수집한다."
        ),
        "escalation_criteria": (
            "기관·부대명 직접 사칭, 내부 배포 정황, 계정입력 유도 시 즉시 보고한다."
        ),
        "lessons_learned": (
            "도메인의 실제 등록가능 영역을 기준으로 판단하고 "
            "서브도메인에 포함된 정상 문자열에 현혹되지 않아야 한다."
        ),
        "owner": "위협정보/보안관제",
        "updated_at": "2026-08-01",
        "keywords": "타이포스쿼팅 하위도메인 하이픈 브랜드사칭"
    },
    {
        "doc_id": "WEB-IR-006",
        "doc_type": "침해사고 사례",
        "title": "검색엔진 광고를 이용한 피싱 유입",
        "attack_category": "Malvertising",
        "summary": (
            "검색 광고 결과를 공식 사이트처럼 노출하여 가짜 로그인 또는 "
            "악성 프로그램 다운로드 페이지로 유도한 사례이다."
        ),
        "detection_points": (
            "검색광고 표시 여부, 공식 사이트와 다른 도메인, 다운로드 유도, "
            "브라우저 검색 유입 경로를 확인한다."
        ),
        "response_steps": (
            "광고 URL 및 최종 URL 차단, 검색 플랫폼 신고, "
            "다운로드 파일 격리 분석과 접속 단말 점검을 수행한다."
        ),
        "evidence_to_collect": (
            "검색 결과 화면, 광고 식별정보, 리퍼러 로그, 최종 URL, "
            "다운로드 파일 해시와 실행 흔적을 수집한다."
        ),
        "escalation_criteria": (
            "악성파일 실행, 관리자 권한 요구, 내부 단말 다수 감염 가능성이 있으면 격상한다."
        ),
        "lessons_learned": (
            "검색 결과 상단 노출은 신뢰 근거가 아니며 공식 북마크 사용 교육이 필요하다."
        ),
        "owner": "침해대응/교육",
        "updated_at": "2026-08-01",
        "keywords": "검색광고 악성광고 다운로드 리퍼러"
    },
    {
        "doc_id": "WEB-SOP-001",
        "doc_type": "대응절차",
        "title": "피싱 URL 초동 분석 절차",
        "attack_category": "Common",
        "summary": (
            "피싱 의심 URL 접수 후 안전하게 식별·분석·차단하기 위한 표준 절차이다."
        ),
        "detection_points": (
            "URL 구조, IP 직접사용, 길이, 단축서비스, 하위도메인, 인증서, "
            "도메인 등록일, DNS, 리다이렉션, 로그인폼을 순차 점검한다."
        ),
        "response_steps": (
            "접수정보 보존 → 격리환경 분석 → URL·도메인·IP 식별 → "
            "차단 → 사용자 영향 확인 → 계정 및 단말 후속조치 → 결과보고 순으로 수행한다."
        ),
        "evidence_to_collect": (
            "원문 URL, 접수시각, 신고자, 화면 캡처, HTTP·DNS·TLS 정보, "
            "접속 로그와 사용자 조치 여부를 확보한다."
        ),
        "escalation_criteria": (
            "실제 계정입력, 악성코드 실행, 중요 자산 접근, 다수 피해자 발생 시 즉시 격상한다."
        ),
        "lessons_learned": (
            "운영 단말에서 직접 접속하지 않고 원문과 분석 결과를 분리 보관한다."
        ),
        "owner": "보안관제",
        "updated_at": "2026-08-01",
        "keywords": "초동분석 URL DNS TLS 차단 증적"
    },
    {
        "doc_id": "WEB-SOP-002",
        "doc_type": "대응절차",
        "title": "피싱 계정 노출 대응 절차",
        "attack_category": "Credential Exposure",
        "summary": (
            "사용자가 피싱 페이지에 계정정보를 입력한 경우의 계정 보호 절차이다."
        ),
        "detection_points": (
            "사용자 진술, 피싱 페이지 입력 여부, 입력시각, 이후 비정상 로그인, "
            "MFA 알림 및 세션 생성 여부를 확인한다."
        ),
        "response_steps": (
            "계정 잠금 또는 비밀번호 초기화, 모든 세션 종료, MFA 재등록, "
            "로그인·메일전달·권한변경 이력 점검, 연관 계정 확대조사를 수행한다."
        ),
        "evidence_to_collect": (
            "로그인 이력, 세션 토큰 기록, MFA 이벤트, 메일 전달규칙, "
            "권한 변경 및 파일 접근 이력을 수집한다."
        ),
        "escalation_criteria": (
            "권한계정 노출, 중요정보 접근, 외부 로그인 성공, 내부 확산 정황 시 격상한다."
        ),
        "lessons_learned": (
            "비밀번호 변경만으로 부족하며 세션과 MFA, 메일 규칙을 함께 초기화해야 한다."
        ),
        "owner": "계정관리/침해대응",
        "updated_at": "2026-08-01",
        "keywords": "계정노출 비밀번호 세션 MFA 로그인"
    },
    {
        "doc_id": "WEB-CHK-001",
        "doc_type": "분석 체크리스트",
        "title": "도메인·DNS·WHOIS 분석 체크리스트",
        "attack_category": "Domain Analysis",
        "summary": (
            "의심 도메인의 생성시점, 소유정보, DNS 인프라와 연관성을 확인하는 체크리스트이다."
        ),
        "detection_points": (
            "도메인 등록일, 등록기관, 네임서버, A·AAAA·MX·TXT 레코드, "
            "동일 IP 호스팅 도메인, 인증서 연관 도메인을 확인한다."
        ),
        "response_steps": (
            "조회 결과를 시간정보와 함께 보존하고, 알려진 정상 도메인 및 "
            "내부 허용목록과 비교한 뒤 IOC 후보를 정리한다."
        ),
        "evidence_to_collect": (
            "WHOIS 원문, DNS 응답, Passive DNS, 인증서 투명성 로그, "
            "조회 시각과 분석자 정보를 수집한다."
        ),
        "escalation_criteria": (
            "군 관련 명칭 사용, 다수 유사 도메인 등록, 동일 인프라의 악성 이력 확인 시 보고한다."
        ),
        "lessons_learned": (
            "등록정보 비공개만으로 악성을 단정하지 말고 다른 증거와 결합한다."
        ),
        "owner": "위협정보",
        "updated_at": "2026-08-01",
        "keywords": "WHOIS DNS PassiveDNS 네임서버 등록일"
    },
    {
        "doc_id": "WEB-CHK-002",
        "doc_type": "분석 체크리스트",
        "title": "웹페이지 행위 분석 체크리스트",
        "attack_category": "Web Behavior",
        "summary": (
            "의심 페이지의 로그인폼, 외부 전송, 다운로드, 스크립트 행위를 확인한다."
        ),
        "detection_points": (
            "폼 전송 대상, 외부 도메인 요청, 난독화 JavaScript, "
            "클립보드 접근, 파일 다운로드, 브라우저 알림 요구를 확인한다."
        ),
        "response_steps": (
            "격리 브라우저에서 네트워크와 DOM 변화를 기록하고 "
            "전송 대상 및 다운로드 파일을 별도 분석한다."
        ),
        "evidence_to_collect": (
            "HAR, 페이지 소스, 스크립트, DOM, 화면녹화, 다운로드 파일, "
            "네트워크 요청과 응답을 수집한다."
        ),
        "escalation_criteria": (
            "계정정보 외부전송, 실행파일 배포, 취약점 악용 스크립트 확인 시 격상한다."
        ),
        "lessons_learned": (
            "정적 화면만으로 판단하지 말고 사용자 입력 전후 행위 변화를 기록한다."
        ),
        "owner": "악성코드/웹분석",
        "updated_at": "2026-08-01",
        "keywords": "HAR DOM JavaScript 폼전송 다운로드"
    },
    {
        "doc_id": "WEB-RPT-001",
        "doc_type": "보고기준",
        "title": "피싱 웹사이트 사고 보고 및 격상 기준",
        "attack_category": "Reporting",
        "summary": (
            "피싱 웹사이트 탐지 결과를 사건으로 전환하고 보고 수준을 결정하기 위한 기준이다."
        ),
        "detection_points": (
            "대상 자산 중요도, 사용자 접속 수, 계정 입력 여부, 악성파일 실행, "
            "외부 유출, 내부 확산 가능성을 평가한다."
        ),
        "response_steps": (
            "초기 사실관계, 탐지 근거, 영향범위, 조치현황, 미확인 사항, "
            "추가 계획을 구분하여 보고한다."
        ),
        "evidence_to_collect": (
            "타임라인, IOC 목록, 영향 사용자, 계정·단말 조치, "
            "스크린샷과 로그 위치를 보고서에 연결한다."
        ),
        "escalation_criteria": (
            "중요계정 또는 중요자산 관련, 다수 피해, 작전 영향, "
            "정보 유출 또는 지속 침투 가능성이 있으면 긴급 격상한다."
        ),
        "lessons_learned": (
            "확인 사실과 추정 내용을 분리하고 미확인 사항을 명시해야 한다."
        ),
        "owner": "상황보고/침해대응",
        "updated_at": "2026-08-01",
        "keywords": "보고 격상 영향도 타임라인 IOC"
    },
    {
        "doc_id": "WEB-LL-001",
        "doc_type": "교훈",
        "title": "피싱 URL 차단 후 재발 방지 교훈",
        "attack_category": "Lessons Learned",
        "summary": (
            "단일 URL 차단 후 유사 도메인과 경로 변형으로 공격이 재발한 사례의 교훈이다."
        ),
        "detection_points": (
            "동일 등록자·네임서버·IP·인증서·페이지 템플릿을 사용하는 "
            "연관 인프라 존재 여부를 확인한다."
        ),
        "response_steps": (
            "연관 IOC 확장, 탐지규칙 보완, 프록시·DNS 차단 동기화, "
            "사용자 안내와 재발 모니터링을 수행한다."
        ),
        "evidence_to_collect": (
            "연관 도메인 목록, 공통 인프라, 페이지 유사도, 재접속 탐지 로그를 수집한다."
        ),
        "escalation_criteria": (
            "차단 후 지속 재발, 다수 변종 생성, 특정 조직을 지속 표적화할 경우 캠페인으로 격상한다."
        ),
        "lessons_learned": (
            "단일 IOC가 아니라 공격 인프라와 행위 패턴 단위로 탐지를 확장해야 한다."
        ),
        "owner": "위협정보/탐지개발",
        "updated_at": "2026-08-01",
        "keywords": "재발 변종 연관IOC 탐지규칙 캠페인"
    }
]

kb_df = pd.DataFrame(knowledge_documents)

# 검색에 사용할 통합 텍스트 생성
search_columns = [
    "doc_type",
    "title",
    "attack_category",
    "summary",
    "detection_points",
    "response_steps",
    "evidence_to_collect",
    "escalation_criteria",
    "lessons_learned",
    "keywords"
]

kb_df["search_text"] = (
    kb_df[search_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

# 센터에서 계속 수정·확장할 수 있도록 CSV로 저장
kb_df.drop(columns=["search_text"]).to_csv(
    "center_phishing_web_rag_knowledge.csv",
    index=False,
    encoding="utf-8-sig"
)

print("센터형 RAG 지식베이스 생성 완료")
print("문서 수:", len(kb_df))
print("저장 파일: center_phishing_web_rag_knowledge.csv")

display(
    kb_df[
        [
            "doc_id",
            "doc_type",
            "title",
            "attack_category",
            "owner",
            "updated_at"
        ]
    ]
)

센터형 RAG 지식베이스 생성 완료
문서 수: 12
저장 파일: center_phishing_web_rag_knowledge.csv


,doc_id,doc_type,title,attack_category,owner,updated_at
0,WEB-IR-001,침해사고 사례,공식 로그인 페이지 위장 계정탈취 사고,Credential Phishing,보안관제/침해대응,2026-08-01
1,WEB-IR-002,침해사고 사례,IP 주소 직접 접속형 피싱 페이지,IP-based Phishing,보안관제,2026-08-01
2,WEB-IR-003,침해사고 사례,단축 URL 및 다단계 리다이렉션 피싱,Redirect Phishing,침해대응,2026-08-01
3,WEB-IR-004,침해사고 사례,HTTPS 인증서를 사용한 정상사이트 위장,HTTPS Phishing,보안관제,2026-08-01
4,WEB-IR-005,침해사고 사례,브랜드 철자 변조 및 하위도메인 위장,Typosquatting,위협정보/보안관제,2026-08-01
5,WEB-IR-006,침해사고 사례,검색엔진 광고를 이용한 피싱 유입,Malvertising,침해대응/교육,2026-08-01
6,WEB-SOP-001,대응절차,피싱 URL 초동 분석 절차,Common,보안관제,2026-08-01
7,WEB-SOP-002,대응절차,피싱 계정 노출 대응 절차,Credential Exposure,계정관리/침해대응,2026-08-01
8,WEB-CHK-001,분석 체크리스트,도메인·DNS·WHOIS 분석 체크리스트,Domain Analysis,위협정보,2026-08-01
9,WEB-CHK-002,분석 체크리스트,웹페이지 행위 분석 체크리스트,Web Behavior,악성코드/웹분석,2026-08-01


In [11]:
# ============================================================
# 11. 임베딩 모델 로드 및 문서 벡터화
# ============================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

kb_embeddings = embedding_model.encode(
    kb_df["search_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("문서 임베딩 크기:", kb_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

문서 임베딩 크기: (12, 384)


In [12]:
# ============================================================
# 12. 임베딩 기반 RAG 검색 함수
# ============================================================

selected_features = [
    "having_IP_Address", "URL_Length", "Shortining_Service",
    "Prefix_Suffix", "having_Sub_Domain", "SSLfinal_State",
    "HTTPS_token", "Abnormal_URL", "Redirect", "age_of_domain",
    "DNSRecord", "web_traffic", "Google_Index"
]


def build_query(row):
    lines = [
        "피싱 웹사이트 머신러닝 탐지 결과",
        f"판정: {row.get('prediction_name')}",
        f"피싱 확률: {row.get('phishing_probability'):.4f}"
    ]

    for feature in selected_features:
        if feature in row.index:
            lines.append(f"{feature}: {row.get(feature)}")

    return "\n".join(lines)


def search_documents(query, top_k=3):
    # Qwen에 넣는 문서 수를 5건에서 3건으로 줄여 속도를 높입니다.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    similarities = kb_embeddings @ query_embedding
    indices = np.argsort(similarities)[::-1][:top_k]

    searched = kb_df.iloc[indices].copy()
    searched["similarity"] = similarities[indices]
    return searched.reset_index(drop=True)


sample_search = search_documents(build_query(top5_df.iloc[0]), top_k=3)
display(sample_search[["doc_id", "doc_type", "title", "attack_category", "similarity"]])


,doc_id,doc_type,title,attack_category,similarity
0,WEB-IR-002,침해사고 사례,IP 주소 직접 접속형 피싱 페이지,IP-based Phishing,0.513741
1,WEB-SOP-001,대응절차,피싱 URL 초동 분석 절차,Common,0.482525
2,WEB-IR-004,침해사고 사례,HTTPS 인증서를 사용한 정상사이트 위장,HTTPS Phishing,0.429282


In [13]:
# ============================================================
# 13. 빠른 Qwen 모델 로드
# ============================================================

# 1.5B 대신 0.5B 모델을 사용해 실습 속도를 크게 줄입니다.
qwen_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("현재 실행 장치:", "cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("주의: CPU에서는 느릴 수 있습니다. Colab 런타임을 T4 GPU로 변경하세요.")

tokenizer = AutoTokenizer.from_pretrained(qwen_name)

if torch.cuda.is_available():
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
else:
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )

qwen_model.eval()
print("Qwen 로드 완료:", qwen_name)
print("모델 실행 장치:", next(qwen_model.parameters()).device)


현재 실행 장치: cpu
주의: CPU에서는 느릴 수 있습니다. Colab 런타임을 T4 GPU로 변경하세요.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen 로드 완료: Qwen/Qwen2.5-0.5B-Instruct
모델 실행 장치: cpu


In [14]:
# ============================================================
# 14. 빠른 Qwen 생성 함수
# ============================================================

def generate_qwen(system_text, user_text, max_new_tokens=420):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 입력 길이를 제한해 긴 RAG 문서 때문에 느려지는 현상을 줄입니다.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2800
    )

    device = next(qwen_model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [15]:
# ============================================================
# 15. RAG 검색 및 빠른 Qwen 보고서 생성
# ============================================================

report_rows = []

for index, row in top5_df.iterrows():
    print(f"[{index + 1}/{len(top5_df)}] {row['event_id']} 분석 중")

    query = build_query(row)
    searched = search_documents(query, top_k=3)

    context_parts = []
    document_details = []

    for rank, (_, doc) in enumerate(searched.iterrows(), start=1):
        # Qwen 입력에는 핵심 필드만 넣어 속도를 줄입니다.
        context_parts.append(
            f"[문서 {rank}]\n"
            f"문서명: {doc['title']}\n"
            f"공격 유형: {doc['attack_category']}\n"
            f"개요: {doc['summary']}\n"
            f"탐지 근거: {doc['detection_points']}\n"
            f"대응절차: {doc['response_steps']}\n"
            f"유사도: {doc['similarity']:.4f}"
        )

        # CSV와 HTML에는 상세 내용을 그대로 보존합니다.
        document_details.append({
            "rank": rank,
            "doc_id": doc["doc_id"],
            "doc_type": doc["doc_type"],
            "title": doc["title"],
            "attack_category": doc["attack_category"],
            "summary": doc["summary"],
            "detection_points": doc["detection_points"],
            "response_steps": doc["response_steps"],
            "evidence_to_collect": doc["evidence_to_collect"],
            "escalation_criteria": doc["escalation_criteria"],
            "lessons_learned": doc["lessons_learned"],
            "owner": doc["owner"],
            "updated_at": doc["updated_at"],
            "similarity": round(float(doc["similarity"]), 4)
        })

    context = "\n\n".join(context_parts)

    user_prompt = f"""
[머신러닝 피싱 웹사이트 탐지 결과]
{query}

[RAG 검색 결과]
{context}

다음 형식으로 간결하게 작성하라.

1. 사건 개요
2. 주요 URL 특징 해석
3. 예상 피싱 유형
4. RAG 검색 근거
5. 초동조치
6. 확보할 증적
7. 최종 판단

각 항목은 2~3문장으로 작성한다.
머신러닝 결과만으로 공격을 확정하지 않는다.
확인된 사실과 추정 내용을 구분한다.
"""

    report = generate_qwen(
        system_text=(
            "너는 사이버보안센터의 피싱 웹사이트 분석관이다. "
            "입력된 탐지값과 검색 문서만 근거로 한국어 보고서를 간결하게 작성한다."
        ),
        user_text=user_prompt,
        max_new_tokens=420
    )

    output_row = row.to_dict()
    output_row.update({
        "top_document_title": searched.iloc[0]["title"],
        "retrieval_similarity": round(float(searched.iloc[0]["similarity"]), 4),
        "retrieved_documents": " | ".join(searched["title"].tolist()),
        "retrieved_document_details": json.dumps(document_details, ensure_ascii=False, indent=2),
        "qwen_report": report
    })
    report_rows.append(output_row)

    # 사건별로 바로 완료 여부를 보여 줍니다.
    print(f"[{index + 1}/{len(top5_df)}] {row['event_id']} 완료")

report_df = pd.DataFrame(report_rows)
report_df.to_csv(
    "exercise1_phishing_rag_reports.csv",
    index=False,
    encoding="utf-8-sig"
)

print("보고서 CSV 저장 완료")
display(report_df[[
    "event_id", "prediction_name", "phishing_probability",
    "top_document_title", "retrieval_similarity"
]])


[1/5] WEB-PHISH-001 분석 중
[1/5] WEB-PHISH-001 완료
[2/5] WEB-PHISH-002 분석 중
[2/5] WEB-PHISH-002 완료
[3/5] WEB-PHISH-003 분석 중
[3/5] WEB-PHISH-003 완료
[4/5] WEB-PHISH-004 분석 중
[4/5] WEB-PHISH-004 완료
[5/5] WEB-PHISH-005 분석 중
[5/5] WEB-PHISH-005 완료
보고서 CSV 저장 완료


,event_id,prediction_name,phishing_probability,top_document_title,retrieval_similarity
0,WEB-PHISH-001,피싱 웹사이트,1.0,IP 주소 직접 접속형 피싱 페이지,0.5137
1,WEB-PHISH-002,피싱 웹사이트,1.0,IP 주소 직접 접속형 피싱 페이지,0.5113
2,WEB-PHISH-003,피싱 웹사이트,1.0,IP 주소 직접 접속형 피싱 페이지,0.4999
3,WEB-PHISH-004,피싱 웹사이트,1.0,IP 주소 직접 접속형 피싱 페이지,0.4967
4,WEB-PHISH-005,피싱 웹사이트,1.0,IP 주소 직접 접속형 피싱 페이지,0.5140


In [16]:
# ============================================================
# 16. Python으로 HTML 대시보드 즉시 생성
# ============================================================

# HTML 전체를 Qwen에게 6,000토큰 생성시키지 않고 Python으로 바로 만듭니다.
# 따라서 이 단계는 거의 즉시 끝납니다.

import html


def safe(value):
    if pd.isna(value):
        return "-"
    return html.escape(str(value))


def probability_level(probability):
    if probability >= 0.90:
        return "매우 높음"
    if probability >= 0.75:
        return "높음"
    return "주의"


cards = []
for _, row in report_df.iterrows():
    try:
        docs = json.loads(row.get("retrieved_document_details", "[]"))
    except Exception:
        docs = []

    doc_blocks = []
    for doc in docs:
        doc_blocks.append(f"""
        <details class="doc">
          <summary>{safe(doc.get('rank'))}위 · {safe(doc.get('title'))} · 유사도 {safe(doc.get('similarity'))}</summary>
          <div class="doc-body">
            <p><b>문서 유형:</b> {safe(doc.get('doc_type'))}</p>
            <p><b>공격 유형:</b> {safe(doc.get('attack_category'))}</p>
            <p><b>개요:</b> {safe(doc.get('summary'))}</p>
            <p><b>탐지 근거:</b> {safe(doc.get('detection_points'))}</p>
            <p><b>대응절차:</b> {safe(doc.get('response_steps'))}</p>
            <p><b>확보 증적:</b> {safe(doc.get('evidence_to_collect'))}</p>
            <p><b>격상 기준:</b> {safe(doc.get('escalation_criteria'))}</p>
            <p><b>교훈:</b> {safe(doc.get('lessons_learned'))}</p>
          </div>
        </details>
        """)

    feature_items = []
    for feature in selected_features:
        if feature in row.index:
            feature_items.append(
                f'<div class="feature"><span>{safe(feature)}</span><b>{safe(row.get(feature))}</b></div>'
            )

    probability = float(row.get("phishing_probability", 0))
    search_text = " ".join([
        str(row.get("event_id", "")),
        str(row.get("prediction_name", "")),
        str(row.get("top_document_title", "")),
        str(row.get("retrieved_documents", "")),
        str(row.get("qwen_report", ""))
    ]).lower()

    cards.append(f"""
    <article class="card" data-prob="{probability}" data-search="{safe(search_text)}">
      <div class="card-head">
        <div>
          <span class="event-id">{safe(row.get('event_id'))}</span>
          <h2>{safe(row.get('prediction_name'))}</h2>
          <p>{safe(row.get('detected_at'))}</p>
        </div>
        <div class="risk">
          <strong>{probability:.1%}</strong>
          <span>{probability_level(probability)}</span>
        </div>
      </div>

      <h3>주요 URL 특징</h3>
      <div class="features">{''.join(feature_items)}</div>

      <h3>RAG 검색 결과</h3>
      <p class="top-doc">최상위 문서: <b>{safe(row.get('top_document_title'))}</b> · 유사도 {safe(row.get('retrieval_similarity'))}</p>
      {''.join(doc_blocks)}

      <details class="report" open>
        <summary>Qwen 분석 보고서</summary>
        <pre>{safe(row.get('qwen_report'))}</pre>
      </details>
    </article>
    """)


generated_html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>피싱 웹사이트 보안관제 대시보드</title>
<style>
* {{ box-sizing: border-box; }}
body {{ margin:0; background:#0b1020; color:#e8edf7; font-family:Arial,'Noto Sans KR',sans-serif; }}
header {{ padding:32px 5%; background:linear-gradient(135deg,#111b38,#172554); border-bottom:1px solid #30436f; }}
header h1 {{ margin:0 0 8px; font-size:28px; }}
header p {{ margin:0; color:#aebbd4; }}
.controls {{ display:flex; gap:12px; flex-wrap:wrap; padding:20px 5%; position:sticky; top:0; background:#0b1020ee; backdrop-filter:blur(8px); z-index:5; }}
input,select {{ padding:12px 14px; border-radius:10px; border:1px solid #344565; background:#111a2d; color:#fff; min-width:220px; }}
main {{ width:min(1200px,90%); margin:0 auto 50px; }}
.card {{ background:#111a2d; border:1px solid #2a3958; border-radius:16px; padding:22px; margin:18px 0; box-shadow:0 12px 32px #0004; }}
.card-head {{ display:flex; justify-content:space-between; gap:20px; align-items:flex-start; }}
.card h2 {{ margin:8px 0; }}
.card h3 {{ margin-top:24px; color:#b9c8e8; }}
.event-id {{ display:inline-block; color:#8fb8ff; font-weight:bold; }}
.risk {{ min-width:120px; text-align:center; padding:14px; border-radius:14px; background:#3b1420; border:1px solid #803047; }}
.risk strong {{ display:block; font-size:25px; color:#ff91aa; }}
.risk span {{ font-size:13px; color:#ffc2d0; }}
.features {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(190px,1fr)); gap:10px; }}
.feature {{ display:flex; justify-content:space-between; gap:10px; background:#0b1325; border:1px solid #293957; padding:10px; border-radius:9px; }}
.feature span {{ color:#9eacc6; font-size:13px; overflow-wrap:anywhere; }}
.doc,.report {{ margin:10px 0; border:1px solid #304363; border-radius:10px; background:#0d1628; }}
summary {{ cursor:pointer; padding:13px; font-weight:bold; }}
.doc-body {{ padding:0 15px 12px; color:#c8d2e5; line-height:1.55; }}
.report pre {{ white-space:pre-wrap; word-break:keep-all; font-family:inherit; line-height:1.7; padding:0 16px 16px; color:#e9edf6; }}
.top-doc {{ color:#b7c4dc; }}
.empty {{ padding:50px; text-align:center; color:#9cabc4; }}
@media(max-width:650px) {{ .card-head {{ flex-direction:column; }} .risk {{ width:100%; }} input,select {{ width:100%; }} }}
</style>
</head>
<body>
<header>
  <h1>피싱 웹사이트 보안관제 대시보드</h1>
  <p>Random Forest 탐지 · 임베딩 RAG 검색 · Qwen 간결 분석</p>
</header>
<div class="controls">
  <input id="search" type="search" placeholder="이벤트·문서·보고서 검색">
  <select id="riskFilter">
    <option value="all">전체 위험도</option>
    <option value="0.90">90% 이상</option>
    <option value="0.75">75% 이상</option>
  </select>
</div>
<main id="cards">{''.join(cards)}</main>
<script>
const search=document.getElementById('search');
const filter=document.getElementById('riskFilter');
function applyFilter() {{
  const q=search.value.toLowerCase().trim();
  const threshold=filter.value==='all' ? 0 : Number(filter.value);
  document.querySelectorAll('.card').forEach(card => {{
    const okText=!q || card.dataset.search.includes(q);
    const okRisk=Number(card.dataset.prob)>=threshold;
    card.style.display=(okText && okRisk) ? 'block' : 'none';
  }});
}}
search.addEventListener('input',applyFilter);
filter.addEventListener('change',applyFilter);
</script>
</body>
</html>"""

with open("exercise1_qwen_phishing_dashboard.html", "w", encoding="utf-8") as file:
    file.write(generated_html)

print("HTML 저장 완료: exercise1_qwen_phishing_dashboard.html")


HTML 저장 완료: exercise1_qwen_phishing_dashboard.html


In [17]:
# ============================================================
# 17. HTML 미리보기 및 결과 다운로드
# ============================================================

display(HTML(generated_html))

print("생성 파일")
print("- exercise1_phishing_top5.csv")
print("- exercise1_phishing_rag_reports.csv")
print("- exercise1_qwen_phishing_dashboard.html")
print("- center_phishing_web_rag_knowledge.csv")

files.download("exercise1_phishing_top5.csv")
files.download("exercise1_phishing_rag_reports.csv")
files.download("exercise1_qwen_phishing_dashboard.html")
files.download("center_phishing_web_rag_knowledge.csv")


생성 파일
- exercise1_phishing_top5.csv
- exercise1_phishing_rag_reports.csv
- exercise1_qwen_phishing_dashboard.html
- center_phishing_web_rag_knowledge.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>